# Showz: retención y adquisición por fuente

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** Los datos no contienen asignación experimental; se retiraron del extracto público las secciones posteriores que no pertenecen al análisis histórico. El LTV del notebook es ingreso observado por comprador, sin margen ni horizonte de vida completo. Su indicador llamado ROMI usa ingresos menos gasto dividido por gasto: no equivale al retorno basado en beneficio incremental. La atribución first touch es una regla descriptiva.


# Sprint 10 - Proyecto Análisis de negocio

## Introduccion

En este proyecto voy a analizar visitas, compras y gastos de marketing para entender el comportamiento de los usuarios, cuánto generan y si el gasto en adquisición se recupera.

## Paso 1. Acceda los datos y prepáralos para el análisis

Primero importo las librerias y cargo los datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
visits = pd.read_csv('/datasets/visits_log_us.csv')
orders = pd.read_csv('/datasets/orders_log_us.csv')
costs = pd.read_csv('/datasets/costs_us.csv')

print(visits.head())
print(orders.head())
print(costs.head())

Dejo los nombres de columnas en un formato más cómodo y convierto fechas a datetime. También calculo la duración de sesión en segundos para tenerla lista.

In [ ]:
# Renombrar columnas (más cómodo para trabajar)
visits = visits.rename(columns={
    'Uid': 'uid',
    'Device': 'device',
    'Start Ts': 'start_ts',
    'End Ts': 'end_ts',
    'Source Id': 'source_id'
})

orders = orders.rename(columns={
    'Uid': 'uid',
    'Buy Ts': 'buy_ts',
    'Revenue': 'revenue'
})

costs = costs.rename(columns={
    'source_id': 'source_id',
    'dt': 'dt',
    'costs': 'costs'
})

# Convertir a datetime
visits['start_ts'] = pd.to_datetime(visits['start_ts'])
visits['end_ts'] = pd.to_datetime(visits['end_ts'])

orders['buy_ts'] = pd.to_datetime(orders['buy_ts'])

costs['dt'] = pd.to_datetime(costs['dt'])

# Duración de sesión (segundos)
visits['session_duration_sec'] = (visits['end_ts'] - visits['start_ts']).dt.total_seconds()

# Revisar info general
visits.info(), orders.info(), costs.info()

Hago una revisión rápida de calidad: fechas, valores negativos en duración, nulos y duplicados obvios.

In [ ]:
print("Duración < 0:", (visits['session_duration_sec'] < 0).sum())
print("Duración = 0:", (visits['session_duration_sec'] == 0).sum())
print("Nulos visitas:\n", visits.isna().sum())
print("Nulos orders:\n", orders.isna().sum())
print("Nulos costs:\n", costs.isna().sum())

**Ejemplos de filas problematicas(negativas y 0)**

In [ ]:
# Ejemplos de sesiones con duración negativa
bad_negative = visits[visits['session_duration_sec'] < 0]
print("Ejemplos duración negativa:")
print(bad_negative.head())

# Ejemplos de sesiones con duración 0
bad_zero = visits[visits['session_duration_sec'] == 0]
print("\nEjemplos duración 0:")
print(bad_zero.head())


**Limpieza de duración de sesión**

Se encontraron 2 sesiones con duración negativa, lo cual indica un registro inconsistente (fin antes del inicio), por lo que se eliminan.
También se encontraron sesiones con duración 0. Estas pueden corresponder a visitas donde el usuario salió de inmediato o a eventos con registro incompleto.
Para evitar sesgos en métricas como duración promedio y distribución de sesiones, se excluyen del análisis de duración.
Sin embargo, las sesiones con duración 0 pueden mantenerse para métricas de conteo (como número de sesiones) si se desea.


**Aplicar limpieza**

In [ ]:
# Quitar duraciones negativas (inconsistentes)
visits_clean = visits[visits['session_duration_sec'] >= 0].copy()

# Dataset para análisis de duración (excluye 0 segundos)
visits_duration = visits_clean[visits_clean['session_duration_sec'] > 0].copy()

print("Filas originales:", len(visits))
print("Filas sin negativas:", len(visits_clean))
print("Filas duración > 0:", len(visits_duration))


Con esta preparación, los datos quedan listos para el análisis: columnas estandarizadas, fechas en formato datetime y una variable de duración de sesión calculada.
Además, se definieron reglas claras para tratar valores inconsistentes (duración negativa) y valores de duración 0 según el tipo de métrica a analizar.


## Paso 2. Haz informes y calcula métricas 

**1. Visitas**

**¿Cuántas personas usan el servicio por día / semana / mes?**

Aquí calculo DAU, WAU y MAU usando usuarios únicos (uid) agrupando por día/semana/mes.

In [ ]:
visits['date'] = visits['start_ts'].dt.date
visits['week'] = visits['start_ts'].dt.to_period('W').astype(str)
visits['month'] = visits['start_ts'].dt.to_period('M').astype(str)

dau = visits.groupby('date')['uid'].nunique()
wau = visits.groupby('week')['uid'].nunique()
mau = visits.groupby('month')['uid'].nunique()

print("DAU (primeros 5):")
print(dau.head())
print("\nWAU (primeros 5):")
print(wau.head())
print("\nMAU (primeros 5):")
print(mau.head())


In [ ]:
plt.figure(figsize=(12,4))
dau.plot()
plt.title('DAU - Usuarios únicos por día')
plt.xlabel('Fecha')
plt.ylabel('Usuarios únicos')
plt.show()

plt.figure(figsize=(12,4))
mau.plot()
plt.title('MAU - Usuarios únicos por mes')
plt.xlabel('Mes')
plt.ylabel('Usuarios únicos')
plt.show()

plt.figure(figsize=(12,4))
wau.plot()
plt.title('WAU - Usuarios únicos por semana')
plt.xlabel('Semana')
plt.ylabel('Usuarios únicos')
plt.xticks(rotation=45)
plt.show()


Al analizar los usuarios únicos por día, semana y mes, se observa que el uso del servicio no es completamente estable en el corto plazo, ya que el DAU presenta variaciones diarias. Sin embargo, al revisar el MAU, se aprecia una base de usuarios más consistente a lo largo del tiempo, lo que indica que, aunque no todos los usuarios visitan el sitio todos los días, sí regresan dentro del mismo mes. Esto sugiere que Showz tiene un nivel aceptable de recurrencia y que el servicio mantiene el interés de los usuarios en el mediano plazo.

En los gráficos se observan picos y caídas pronunciadas en ciertos periodos.
Estos cambios pueden estar relacionados con campañas de marketing, eventos puntuales,
problemas de tracking o estacionalidad propia del negocio.
Sin información adicional, no es posible atribuirlos a una causa específica,
pero es importante considerarlos al interpretar las tendencias generales.


**¿Cuántas sesiones hay por día?**

Una sesión es una fila en visits. Aquí cuento sesiones por día (no usuarios).

In [ ]:
sessions_per_day = visits.groupby('date')['uid'].count()

plt.figure(figsize=(12,4))
sessions_per_day.plot()
plt.title('Sesiones por día')
plt.xlabel('Fecha')
plt.ylabel('Número de sesiones')
plt.show()

sessions_per_day.describe()


El número de sesiones complementa el análisis de usuarios únicos,
ya que refleja el nivel de interacción total con el sitio,
independientemente de cuántos usuarios distintos lo visiten.


**¿Cuál es la duración de cada sesión?**

Miro la distribución de duración. Para evitar que valores extremos distorsionen, también reviso percentiles.

In [ ]:
visits_duration['session_duration_sec'].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


In [ ]:
plt.figure(figsize=(10,4))
visits_duration['session_duration_sec'].hist(bins=50)
plt.title('Distribución de duración de sesión (segundos)')
plt.xlabel('Segundos')
plt.ylabel('Frecuencia')
plt.show()


La distribución de la duración de las sesiones muestra que la mayoría de las visitas son relativamente cortas, mientras que solo una pequeña parte de los usuarios permanece mucho tiempo en el sitio.

**¿Con qué frecuencia los usuarios regresan?**

Para entender “qué tanto regresan”, uso cohortes por mes de primera visita y veo si vuelven en meses posteriores (retención por mes).

In [ ]:
# mes de cada visita
visits['visit_month'] = visits['start_ts'].dt.to_period('M')

# primera visita por usuario
first_visit = visits.groupby('uid')['visit_month'].min().reset_index()
first_visit.columns = ['uid', 'first_visit_month']

visits = visits.merge(first_visit, on='uid', how='left')

# "edad" del usuario en meses desde su primer mes
visits['cohort_age'] = (visits['visit_month'] - visits['first_visit_month']).apply(lambda x: x.n)

cohort_counts = (
    visits.groupby(['first_visit_month', 'cohort_age'])['uid']
    .nunique()
    .reset_index()
)

cohort_sizes = (
    visits.groupby('first_visit_month')['uid']
    .nunique()
    .rename('cohort_size')
)

retention = cohort_counts.pivot_table(
    index='first_visit_month',
    columns='cohort_age',
    values='uid'
).div(cohort_sizes, axis=0)

retention.head()


In [ ]:
plt.figure(figsize=(12,5))
for i, row in retention.head(6).iterrows():
    plt.plot(retention.columns, row.values, marker='o', label=str(i))
plt.title('Retención por cohorte (primeras 6 cohortes)')
plt.xlabel('Meses desde la primera visita')
plt.ylabel('Retención (proporción)')
plt.legend()
plt.show()


El análisis de retención por cohortes muestra que la mayor parte de los usuarios regresa principalmente en los primeros meses después de su primera visita, y luego la actividad disminuye con el tiempo.

**2. Ventas**

**¿Cuándo empieza la gente a comprar?**

La idea aquí es comparar cuánto tarda un usuario en hacer su primera compra desde su primera visita. Si compra el mismo día: “Conversion 0d”, si compra al día siguiente: “Conversion 1d”, etc.

In [ ]:
# primera visita y primera compra por usuario
first_visit_ts = visits.groupby('uid')['start_ts'].min()
first_buy_ts = orders.groupby('uid')['buy_ts'].min()

conversion = pd.concat([first_visit_ts, first_buy_ts], axis=1)
conversion.columns = ['first_visit_ts', 'first_buy_ts']
conversion = conversion.dropna()

conversion['days_to_buy'] = (conversion['first_buy_ts'].dt.date - conversion['first_visit_ts'].dt.date).dt.days

conversion['conversion_group'] = conversion['days_to_buy'].apply(lambda x: f"Conversion {x}d" if x <= 30 else "Conversion 31d+")
conversion['days_to_buy'].describe()


In [ ]:
conv_counts = conversion['conversion_group'].value_counts().sort_index()

plt.figure(figsize=(14,4))
conv_counts.plot(kind='bar')
plt.title('Tiempo a primera compra (grupos)')
plt.xlabel('Grupo')
plt.ylabel('Usuarios')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()



La mayoría de los usuarios que realizan una compra lo hacen poco tiempo después de su primera visita, especialmente dentro de los primeros días. Esto indica que el proceso de conversión es relativamente rápido y que muchos usuarios llegan al sitio con una intención clara de compra. Sin embargo, también existe un grupo que tarda más en convertir, lo que podría aprovecharse con campañas de remarketing.

**¿Cuántos pedidos hacen en un periodo?**

Aquí saco pedidos por usuario por mes, para ver si compran una vez y se van o si repiten.

In [ ]:
orders['order_month'] = orders['buy_ts'].dt.to_period('M').astype(str)

orders_per_user_month = (
    orders.groupby(['order_month', 'uid'])['revenue']
    .count()
    .reset_index(name='orders_cnt')
)

orders_cnt_month = orders_per_user_month.groupby('order_month')['orders_cnt'].mean()

plt.figure(figsize=(12,4))
orders_cnt_month.plot(marker='o')
plt.title('Promedio de pedidos por usuario (por mes)')
plt.xlabel('Mes')
plt.ylabel('Pedidos promedio')
plt.show()

orders_cnt_month.describe()


El promedio de pedidos por usuario muestra que la mayoría de los clientes realiza pocas compras en el tiempo analizado. Esto sugiere que el comportamiento dominante es el de compras puntuales más que compras frecuentes, algo esperado en un negocio de venta de entradas.

**¿Cuál es el tamaño promedio de compra?**

In [ ]:
aov_by_month = orders.groupby('order_month')['revenue'].mean()

plt.figure(figsize=(12,4))
aov_by_month.plot(marker='o')
plt.title('Ticket promedio (AOV) por mes')
plt.xlabel('Mes')
plt.ylabel('Revenue promedio por pedido')
plt.show()

aov_by_month.describe()


El ticket promedio se mantiene relativamente estable a lo largo del tiempo, con algunas variaciones entre meses. Esto indica que el monto que los usuarios están dispuestos a pagar por pedido no cambia de forma drástica, por lo que el crecimiento de ingresos depende más del número de compradores y pedidos que de un aumento en el valor de cada compra.  


**¿Cuánto dinero traen?**

Para LTV, hago cohortes por mes de primera compra. Luego calculo ingreso acumulado por usuario en cada cohorte.

In [ ]:
orders['buy_month'] = orders['buy_ts'].dt.to_period('M')

first_buy_month = orders.groupby('uid')['buy_month'].min().reset_index()
first_buy_month.columns = ['uid', 'first_buy_month']

orders_ = orders.merge(first_buy_month, on='uid', how='left')
orders_['cohort_age'] = (orders_['buy_month'] - orders_['first_buy_month']).apply(lambda x: x.n)

cohort_revenue = (
    orders_.groupby(['first_buy_month', 'cohort_age'])['revenue']
    .sum()
    .reset_index()
)

cohort_buyers = (
    orders_.groupby('first_buy_month')['uid']
    .nunique()
    .rename('buyers')
)

ltv = cohort_revenue.pivot_table(
    index='first_buy_month',
    columns='cohort_age',
    values='revenue'
).div(cohort_buyers, axis=0)

ltv_cum = ltv.cumsum(axis=1)
ltv_cum.head()


In [ ]:
plt.figure(figsize=(12,5))
for i, row in ltv_cum.head(6).iterrows():
    plt.plot(ltv_cum.columns, row.values, marker='o', label=str(i))
plt.title('LTV acumulado por cohorte (primeras 6 cohortes)')
plt.xlabel('Meses desde la primera compra')
plt.ylabel('LTV acumulado')
plt.legend()
plt.show()


El análisis de LTV por cohortes muestra que los ingresos se acumulan principalmente en los primeros meses después de la primera compra. A partir de ese punto, el crecimiento del LTV se desacelera, lo que refuerza la idea de que la mayoría de los clientes compra pocas veces.

**3. Marketing**

**¿Cuánto dinero se gastó?**

In [ ]:
costs['month'] = costs['dt'].dt.to_period('M').astype(str)

total_costs = costs['costs'].sum()
print("Gasto total:", total_costs)

costs_by_source = costs.groupby('source_id')['costs'].sum().sort_values(ascending=False)
print(costs_by_source.head(10))

costs_by_month = costs.groupby('month')['costs'].sum()

plt.figure(figsize=(12,4))
costs_by_month.plot(marker='o')
plt.title('Gasto de marketing por mes')
plt.xlabel('Mes')
plt.ylabel('Costos')
plt.show()


**CAC por fuente**

Para CAC necesito: cuánto gasté por fuente / cuántos compradores traje por esa fuente.
Voy a atribuir la fuente usando la primera visita del usuario (first touch), que es una forma estándar cuando no hay atribución más compleja.

In [ ]:
# Fuente de adquisición por usuario: primera visita
first_touch = visits.sort_values('start_ts').groupby('uid').first().reset_index()
user_source = first_touch[['uid', 'source_id']]

# Usuarios que compraron (buyers)
buyers = orders['uid'].unique()
buyers_df = pd.DataFrame({'uid': buyers}).merge(user_source, on='uid', how='left')

buyers_by_source = buyers_df.groupby('source_id')['uid'].nunique()

# Costos totales por fuente
costs_total_by_source = costs.groupby('source_id')['costs'].sum()

cac_by_source = (costs_total_by_source / buyers_by_source).replace([np.inf, -np.inf], np.nan).dropna()
cac_by_source = cac_by_source.sort_values()

cac_by_source.head(10)


In [ ]:
plt.figure(figsize=(12,4))
cac_by_source.plot(kind='bar')
plt.title('CAC por fuente (first touch)')
plt.xlabel('Source ID')
plt.ylabel('CAC')
plt.xticks(rotation=0)
plt.show()


**ROMI por fuente**

ROMI lo calculo como beneficio bruto / gastos. Para ingresos por fuente, asigno revenue a la fuente de adquisición del usuario (first touch).

In [ ]:
orders_with_source = orders.merge(user_source, on='uid', how='left')
revenue_by_source = orders_with_source.groupby('source_id')['revenue'].sum()

romi_by_source = ((revenue_by_source - costs_total_by_source) / costs_total_by_source)
romi_by_source = romi_by_source.replace([np.inf, -np.inf], np.nan).dropna().sort_values(ascending=False)

romi_by_source.head(10)


In [ ]:
plt.figure(figsize=(12,4))
romi_by_source.plot(kind='bar')
plt.title('ROMI por fuente (first touch)')
plt.xlabel('Source ID')
plt.ylabel('ROMI')
plt.xticks(rotation=0)
plt.show()


El análisis de gastos de marketing, CAC y ROMI muestra diferencias claras entre las fuentes.
Algunas fuentes combinan un CAC bajo con un ROMI positivo,
lo que las convierte en candidatas prioritarias para inversión futura.
En contraste, las fuentes con ROMI negativo no logran recuperar el gasto,
por lo que deberían optimizarse o reducir su presupuesto.
Estas métricas permiten orientar la inversión hacia fuentes más rentables
y maximizar el retorno del gasto en marketing.


## Paso 3. Priorización de hipótesis (ICE y RICE)

**3.1 Formulación de hipótesis**

En esta sección se proponen varias hipótesis orientadas a mejorar las métricas clave del negocio, principalmente la conversión y los ingresos.
Para priorizarlas, se utilizan los métodos ICE y RICE, que permiten evaluar el impacto potencial de cada hipótesis considerando diferentes criterios.

**3.2 Tabla de hipótesis**

In [ ]:
hypotheses = pd.DataFrame({
    'hypothesis': [
        'Reducir el número de pasos en el proceso de compra aumentará la conversión',
        'Mostrar recomendaciones personalizadas incrementará el tamaño promedio del pedido',
        'Optimizar la versión móvil mejorará la conversión en dispositivos móviles',
        'Lanzar campañas de remarketing aumentará la tasa de compra de usuarios recurrentes'
    ],
    'reach': [8, 6, 7, 5],
    'impact': [9, 7, 8, 6],
    'confidence': [7, 6, 6, 5],
    'effort': [6, 7, 5, 4]
})

hypotheses


**3.3 Cálculo de ICE**

El método ICE prioriza las hipótesis considerando su impacto esperado, el nivel de confianza y el esfuerzo requerido.

In [ ]:
hypotheses['ICE'] = (
    hypotheses['impact'] * hypotheses['confidence'] / hypotheses['effort']
)

hypotheses.sort_values(by='ICE', ascending=False)
